# 01. Hash maps

**Tier:** Deep. **Why you'd use this:** A dict answers "have I seen this value?" in one step. For a target sum, the complement of a value is `target - value`.

**After this lesson you can:** look up a complement in a dict while you scan once; return an index you stored earlier.
About 20 minutes. **Uses:** [Dicts](../../course/04-collections/03-dicts-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. How fast is a dict lookup for a key that is present?
2. Why store an index in the dict instead of only a boolean "seen" flag when you need a pair of indexes?

<details><summary>Answers</summary>

For 01 first unique lesson: skim [Dicts](../../course/04-collections/03-dicts-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb) if needed.
</details>


## Recognize it

You scan a list and need the first index where a value shows up again. The prompt talks about "already seen", "first duplicate", or "return the earlier index when a value repeats".

Signals: you need a yes/no answer about a value you met before, and you care about an index you stored earlier. A look-alike that is not this pattern: counting how many times each value appears for a frequency report — that still uses a dict, but the answer is the counts, not a stored index from an earlier pass.


## Why it works this way

A nested scan asks, for each index, whether any earlier index holds the same value. That restarts from the front every time. On a list of length `n`, the number of comparisons grows like `1 + 2 + … + (n - 1)`, which is proportional to `n²`. The dict version pays for one hash lookup per value instead. Average lookup cost stays near constant when keys hash well, so the whole pass stays proportional to `n`. See [Dicts](../../internals/04-dicts/01-hash-table-lesson.ipynb) for why a hash table turns an equality search into a bucket probe.

What the dict stores is the contract. Here the value is the first index where that number appeared. On a hit you return that stored index, not the index you are standing on. That matches the sibling question: "where did this value first show up before it repeated?". If you only needed a boolean "have I seen it?", a set would be enough. You keep a dict because the answer is a remembered index.

Order matters for correctness. You look up before you write. If you write first, the current index becomes the stored one and a value can pair with itself on the same step. The invariant after each step is: every value to the left of the cursor has exactly one entry, its earliest index. Memory grows with the number of distinct values. In the worst case that is `n` entries. Time is one pass plus constant work per element when hashing behaves.

A sorted two-pointer search also finds a pair under a different question, but it needs sorted order and answers "does any pair sum to a target?", not "where did this value first appear?". The hash map fits when the list is unsorted and the lookup key is the value itself.


## When to use it

Use a dict of first indexes when the prompt asks for an earlier position of a value you meet again, or more generally when a one-pass lookup of something you stored is cheaper than a nested scan. Prefer a set when you only need presence. Prefer two pointers on a sorted list when the question is about a pair sum and extra memory must stay constant. Prefer a plain double loop only for tiny inputs where the quadratic cost does not matter and the code must stay obvious.


## Brute force, and why it's slow

Count the comparisons. For each index, scan every earlier index looking for the same value.


In [1]:
def first_repeat_slow(nums):
    comparisons = 0
    for index, value in enumerate(nums):
        for earlier in range(index):
            comparisons += 1
            if nums[earlier] == value:
                return earlier, comparisons
    return -1, comparisons


for size in (50, 200, 800):
    data = list(range(size - 1)) + [size // 2]
    print(size, first_repeat_slow(data))


50 (25, 1202)
200 (100, 19802)
800 (400, 319202)


## Key insight

Store each value's first index in a dict as you walk. The first time `value in seen` is true, that stored index is the answer — one pass, no nested scan.


## Worked example

Return the earlier index of the first value that appears twice, or `-1` when every value is unique.


In [2]:
def first_repeat(nums: list[int]) -> int:
    # 1. Remember where each value first appeared.
    seen = {}
    for index, value in enumerate(nums):
        # 2. A hit means this value already has an index.
        if value in seen:
            return seen[value]
        # 3. Miss: record this as the first sighting.
        seen[value] = index
    return -1


print(first_repeat([3, 1, 3, 2]))
print(first_repeat([1, 2, 3]))
print(first_repeat([7, 7]))


0
-1
0


## Trace

`[3, 1, 3, 2]`:

| index | value | seen before step | action |
|---|---|---|---|
| 0 | 3 | `{}` | store `3 → 0` |
| 1 | 1 | `{3: 0}` | store `1 → 1` |
| 2 | 3 | `{3: 0, 1: 1}` | hit → return **0** |


## Cost

One pass over the list: time grows with `n`. The dict holds one entry per distinct value, so extra memory grows with the number of unique numbers. That beats the quadratic nested scan because each value is compared through a hash lookup instead of a fresh left-side walk.


## Variants

Not solved here: [Two Sum](../../challenges/problems/01-two-sum.ipynb), [Group Anagrams](../../challenges/problems/02-group-anagrams.ipynb), [Top K Frequent Elements](../../challenges/problems/15-top-k-frequent-elements.ipynb).


## Common mistake

This bug is silent. The function returns the later index — the one you are standing on — instead of the first sighting. There is no traceback. The print is `2`, and the answer is `0`.


In [3]:
def first_repeat_wrong(nums):
    seen = {}
    for index, value in enumerate(nums):
        if value in seen:
            return index  # later index, not the first
        seen[value] = index
    return -1


print(first_repeat_wrong([3, 1, 3, 2]))


2


The fix returns the stored index. `seen[value]` is the earlier position. The buggy cell returns `2` for `[3, 1, 3, 2]` because it keeps overwriting the first sighting.


In [4]:
print(first_repeat([3, 1, 3, 2]))


0


## Python idioms

`enumerate` pairs each value with its index in one loop. `.get` is useful when a miss should become a default instead of a branch, but here a plain `in` check keeps the "already seen" path obvious.


In [5]:
def first_repeat_get(nums: list[int]) -> int:
    seen = {}
    for index, value in enumerate(nums):
        earlier = seen.get(value)
        if earlier is not None:
            return earlier
        seen[value] = index
    return -1


print(first_repeat_get([3, 1, 3, 2]))


0


## Recap

- A dict stores the first index of each value.
- Look up before you store, or a value pairs with itself.
- The first hit's stored index is the answer.

Next: [practice](01-first-unique-practice.ipynb).
